# Results for model: meta/llama3-70b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# Load 'train.parquet' using Polars
df = pl.read_parquet('./jane_street_data/train.parquet')

# Calculate global TOP_QUANTILE (top 15%) of 'feature_00' relative to 'responder_6' across rolling batches of 'date_id'
df['TOP_QUANTILE'] = df.groupby('date_id')['feature_00'].transform(lambda x: x > x.quantile(0.85))

# Prepare data for training
X = df.drop(['responder_6'], axis=1)
y = df['responder_6']

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost Regressor on the target 'responder_6'
xgb_regressor = XGBRegressor()
xgb_regressor.fit(X_train, y_train)

# Evaluate the model on the validation set
y_pred = xgb_regressor.predict(X_val)
eval_result = xgb_regressor.evaluate(X_val, y_val)
print(f'Validation RMSE: {eval_result:.4f}')